# [AI SIEM] AI 에이전트 공격 탐지 모델 학습/평가 (세션 단위 교차검증)

`03_train_eval.py`를 Colab에서 실행하는 노트북입니다. 학습 로직은 스크립트 하나에만 들어 있어서, PC에서 돌린 결과와 Colab 결과가 항상 같습니다.

**기존 노트북(`ai_siem_rf_colab.ipynb`)과 달라진 점**
- 같은 세션의 창이 학습과 검증에 동시에 들어가지 않도록 세션 단위로 교차검증합니다. 기존 방식은 데이터 누수가 있었습니다.
- 학습에 쓰지 않은 공격 도구로 검증해서, 처음 보는 공격 도구에도 통하는지(일반화) 평가합니다.
- 기본값 0.5 대신, 오탐률 목표(2.5%)를 만족하도록 임계값을 정합니다.
- 단순 규칙과 IsolationForest를 베이스라인으로 두고 RandomForest와 비교합니다.
- 피처 하나만으로 정상과 공격이 갈리는 지름길(shortcut) 피처가 있으면 경고합니다.
- TTD를 공격 세션이 시작된 시점부터 첫 경보까지의 시간으로 계산합니다.

## 1. 파일 업로드
`ml/src/원유찬/03_train_eval.py`와 `ml/data/features/*_features.csv` 파일들을 **한 번에** 선택해서 올리세요.

In [ ]:
import os, shutil
from google.colab import files
uploaded = files.upload()
os.makedirs('features', exist_ok=True)
for name in uploaded:
    if name.endswith('_features.csv'):
        shutil.move(name, os.path.join('features', name))
print('스크립트:', os.path.exists('03_train_eval.py'))
print('피처 파일:', sorted(os.listdir('features')))

## 2. 학습 및 평가
옵션: `--target-fpr 0.025`(오탐률 목표), `--alert-k 2`(연속 몇 개 창을 탐지해야 경보를 낼지), `--exclude-features unique_ua_count`(특정 피처 제외)

In [ ]:
!python 03_train_eval.py --features features --out-dir models --target-fpr 0.025 --alert-k 2

## 3. 그래프

In [ ]:
from IPython.display import Image, display
for f in ['models/cv_curves.png', 'models/permutation_importance.png']:
    if os.path.exists(f):
        display(Image(f))

## 4. 모델/리포트 다운로드

In [ ]:
for f in ['models/attack_detector_rf.joblib', 'models/eval_report.json']:
    if os.path.exists(f):
        files.download(f)